In [1]:
!pip install faster-whisper sentence-transformers faiss-cpu transformers accelerate

In [2]:
from google.colab import files
uploaded = files.upload()

Saving lecture.mp4 to lecture (2).mp4


In [3]:
!apt-get install ffmpeg
!ffmpeg -i lecture.mp4 -q:a 0 -map a lecture.mp3

Streaming output truncated to the last 5000 lines.
[aac @ 0x58013ff3c3c0] Number of bands (44) exceeds limit (42).
Error while decoding stream #0:1: Invalid data found when processing input
[aac @ 0x58013ff3c3c0] channel element 3.10 is not allocated
Error while decoding stream #0:1: Invalid data found when processing input
[aac @ 0x58013ff3c3c0] channel element 2.12 is not allocated
Error while decoding stream #0:1: Invalid data found when processing input
[aac @ 0x58013ff3c3c0] Sample rate index in program config element does not match the sample rate index configured by the container.
[aac @ 0x58013ff3c3c0] Inconsistent channel configuration.
[aac @ 0x58013ff3c3c0] get_buffer() failed
Error while decoding stream #0:1: Invalid argument
[aac @ 0x58013ff3c3c0] Sample rate index in program config element does not match the sample rate index configured by the container.
[aac @ 0x58013ff3c3c0] Inconsistent channel configuration.
[aac @ 0x58013ff3c3c0] get_buffer() failed
Error while decod

In [4]:
from faster_whisper import WhisperModel

model = WhisperModel("base")
segments, _ = model.transcribe("lecture.mp3")

transcript = " ".join([seg.text for seg in segments])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunks = chunk_text(transcript)

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedder.encode(chunks)

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    return [chunks[i] for i in indices[0]]

In [8]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 🔥 Proper 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix padding (important)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("✅ Mistral loaded successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Mistral loaded successfully!


In [10]:
def generate_text(prompt, max_tokens=300):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [11]:
def generate_notes(context):
    prompt = f"""
You are an expert professor.

Generate structured lecture notes from the following content:

Include:
1. Key Concepts
2. Summary
3. Insights (deep understanding, connections, applications)
4. Examples (if possible)

Lecture Content:
{context}
"""
    return generate_text(prompt, 300)

In [12]:
def reflect(notes):
    critique_prompt = f"""
You are a strict reviewer.

Critique the following notes:
- What is missing?
- What is unclear?
- What can be improved?

Notes:
{notes}
"""

    critique = generate_text(critique_prompt, 200)

    improve_prompt = f"""
Improve the following notes using this critique.

Notes:
{notes}

Critique:
{critique}

Return improved structured notes.
"""

    improved = generate_text(improve_prompt, 300)

    return improved

In [13]:
query = "Explain key concepts from the lecture"

context = " ".join(retrieve(query))  # your FAISS retriever

notes = generate_notes(context)

final_notes = reflect(notes)

print("\n========== FINAL NOTES ==========\n")
print(final_notes)


========== FINAL NOTES ==========


Improve the following notes using this critique.

Notes:

You are an expert professor.

Generate structured lecture notes from the following content:

Include:
1. Key Concepts
2. Summary
3. Insights (deep understanding, connections, applications)
4. Examples (if possible)

Lecture Content:
does for a large language model. A standard large language model like GPT, Claude or Geminar is like a student who only has what they memorize during training. They are smart. They can reason, they can write, they can explain things but their knowledge has a cut off date and most importantly they have no idea what's in your documents, what's in your company's databases and what is in your internal knowledge base. Now Rag with stands for retrieval augmented generation fixes that. So instead of relying purely on what the model memorized, Rag gives it the ability to go look up things first. Pull in relevant information and then generate an answer that's grounded in t